#Task 3: Symmetric vs. Asymmetric INT8 Quantization


##Objective

Implement and compare symmetric and asymmetric INT8 quantization. Understand why weights (centered around zero) and activations (often non-negative) benefit from different methods.


In [1]:
#import the req libs

import numpy as np

In [2]:
#inputs given

weights = np.array([
    [-1.8, -0.9, 0.0, 0.7, 1.5],
    [-2.4, -0.3, 0.2, 1.1, 2.0]
], dtype=np.float32)

activations = np.array([
    [0.0, 0.3, 0.8, 1.4, 2.1],
    [0.1, 0.6, 1.0, 1.8, 3.2]
], dtype=np.float32)

Quantized range: [-127, 127]. Zero point is always 0. Scale is based on the maximum absolute value.

scale = max(|x_min|, |x_max|) / 127

q = round(x / scale)

clip to [-127, 127], return as np.int8

In [4]:
#PART A Symmetric int8 quantization

def symmetric_quantize(tensor):

    # Return: (quantized_tensor, scale, zero_point=0)

    max_abs = np.max(np.abs(tensor))

    # Handle zero tensor coz if div by zero error
    if max_abs == 0:
        return tensor.astype(np.int8), 1.0, 0

    scale = max_abs / 127
    quantized = np.round(tensor / scale)
    quantized = np.clip(quantized, -127, 127)

    return quantized.astype(np.int8), scale, 0


Quantized range: [-128, 127]. Both scale and zero point are calculated from the data range.

scale = (x_max - x_min) / 255

zero_point = round(-128 - x_min / scale)

q = round(x / scale) + zero_point

clip zero_point and q to [-128, 127], return as np.int8

In [6]:
#PART B Asymmetric
def asymmetric_quantize(tensor):
    # Return: (quantized_tensor, scale, zero_point)

    x_min = np.min(tensor)
    x_max = np.max(tensor)

    if x_max == x_min:
        return tensor.astype(np.int8), 1.0, 0

    scale = (x_max - x_min) / 255
    zero_point = np.round(-128 - (x_min / scale))
    zero_point = np.clip(zero_point, -128, 127)
    quantized = np.round(tensor / scale) + zero_point
    quantized = np.clip(quantized, -128, 127)

    return quantized.astype(np.int8), scale, int(zero_point)

Part C: Dequantization

x_dequantized = (q - zero_point) * scale


In [8]:
#PART C - Dequantize

def dequantize(quantized_tensor, scale, zero_point):

    # Return: dequantized float tensor

    dequantized = quantized_tensor.astype(np.float32)
    dequantized = dequantized - zero_point
    dequantized = dequantized * scale

    return dequantized


PART D,E : Evalute metrics

In [20]:
def evaluate_quantization(original_tensor, quantized_tensor, dequantized_tensor):


    error = original_tensor - dequantized_tensor

    mae = np.mean(np.abs(error))
    mse = np.mean(error ** 2)
    max_error = np.max(np.abs(error))

    # Count saturated values
    sat_min = np.sum(
        (quantized_tensor == -128) |
        (quantized_tensor == -127)
    )

    sat_max = np.sum(quantized_tensor == 127)

    sat_total = sat_min + sat_max

    return {
        "MAE": mae,
        "MSE": mse,
        "Max Error": max_error,
        "Sat(min)": sat_min,
        "Sat(max)": sat_max,
        "Sat(total)": sat_total
    }

In [21]:
def compare_quantization(name, tensor):
    print(name,":")


    sym_q, sym_scale, sym_zp = symmetric_quantize(tensor)
    sym_deq = dequantize(sym_q, sym_scale, sym_zp)
    sym_metrics = evaluate_quantization(
        tensor,
        sym_q,
        sym_deq
    )

    # Asymmetric Quantization
    asym_q, asym_scale, asym_zp = asymmetric_quantize(tensor)
    asym_deq = dequantize(
        asym_q,
        asym_scale,
        asym_zp
    )

    asym_metrics = evaluate_quantization(
        tensor,
        asym_q,
        asym_deq
    )

    # Print Results

    print("\nOriginal Tensor")
    print(tensor)

    print("\nSymmetric Quantized (INT8)")
    print(sym_q)

    print("\nSymmetric Dequantized (Float)")
    print(sym_deq)

    print("\nAsymmetric Quantized (INT8)")
    print(asym_q)

    print("\nAsymmetric Dequantized (Float)")
    print(asym_deq)


    # Metrics Table

    print("\nComparison")

    print("----------------------------------------------------------------------------")

    print(f"Symmetric Scale      : {sym_scale}")
    print(f"Symmetric Zero Point : {sym_zp}")
    print(f"Symmetric MAE        : {sym_metrics['MAE']}")
    print(f"Symmetric MSE        : {sym_metrics['MSE']}")
    print(f"Symmetric Max Error  : {sym_metrics['Max Error']}")
    print(f"Symmetric Sat(min)   : {sym_metrics['Sat(min)']}")
    print(f"Symmetric Sat(max)   : {sym_metrics['Sat(max)']}")
    print(f"Symmetric Sat(total) : {sym_metrics['Sat(total)']}")

    print("----------------------------------------------------------------------------")

    print(f"Asymmetric Scale      : {asym_scale}")
    print(f"Asymmetric Zero Point : {asym_zp}")
    print(f"Asymmetric MAE        : {asym_metrics['MAE']}")
    print(f"Asymmetric MSE        : {asym_metrics['MSE']}")
    print(f"Asymmetric Max Error  : {asym_metrics['Max Error']}")
    print(f"Asymmetric Sat(min)   : {asym_metrics['Sat(min)']}")
    print(f"Asymmetric Sat(max)   : {asym_metrics['Sat(max)']}")
    print(f"Asymmetric Sat(total) : {asym_metrics['Sat(total)']}")

    print("---------------------------------------------------------------------------")
    print("---------------------------------------------------------------------------")
    print()

In [22]:
compare_quantization(
    "Weights",
    weights
)

compare_quantization(
    "Activations",
    activations
)

Weights :

Original Tensor
[[-1.8 -0.9  0.   0.7  1.5]
 [-2.4 -0.3  0.2  1.1  2. ]]

Symmetric Quantized (INT8)
[[ -95  -48    0   37   79]
 [-127  -16   11   58  106]]

Symmetric Dequantized (Float)
[[-1.7952756  -0.9070866   0.          0.6992126   1.4929134 ]
 [-2.4        -0.3023622   0.20787401  1.096063    2.0031495 ]]

Asymmetric Quantized (INT8)
[[ -93  -41   11   52   98]
 [-128   -6   23   75  127]]

Asymmetric Dequantized (Float)
[[-1.7945098  -0.8972549   0.          0.707451    1.5011765 ]
 [-2.3984313  -0.29333332  0.20705882  1.1043137   2.0015686 ]]

Comparison
----------------------------------------------------------------------------
Symmetric Scale      : 0.018897637724876404
Symmetric Zero Point : 0
Symmetric MAE        : 0.003700774861499667
Symmetric MSE        : 2.1637999452650547e-05
Symmetric Max Error  : 0.007874011993408203
Symmetric Sat(min)   : 1
Symmetric Sat(max)   : 0
Symmetric Sat(total) : 1
-------------------------------------------------------------

| Tensor | Method | Scale | Zero Pt | MAE | MSE | Max Err | Sat(min) | Sat(max) | Sat(total) |
|--------|--------|------:|--------:|------:|------:|------:|---------:|---------:|-----------:|
| Weights | Symmetric | 0.018898 | 0 | 0.003701 | 0.00002164 | 0.007874 | 1 | 0 | 1 |
| Weights | Asymmetric | 0.017255 | 11 | 0.003804 | 0.00002124 | 0.007451 | 1 | 1 | 2 |
| Activations | Symmetric | 0.025197 | 0 | 0.005276 | 0.00004483 | 0.011024 | 0 | 1 | 1 |
| Activations | Asymmetric | 0.012549 | -128 | 0.002627 | 0.00001112 | 0.005490 | 1 | 1 | 2 |

## Observations

- Symmetric quantization performed well for the weight tensor because the values were centered around zero.
- Asymmetric quantization produced lower reconstruction error for the activation tensor since the values were mostly non-negative.


In [24]:
#PART F : Outlier tensor

outlier_tensor = np.array(
    [-0.5, -0.2, 0.0, 0.3, 0.7, 12.0],
    dtype=np.float32
)

In [30]:
compare_quantization(
    "With Outlier",
    outlier_tensor
)

With Outlier :

Original Tensor
[-0.5 -0.2  0.   0.3  0.7 12. ]

Symmetric Quantized (INT8)
[ -5  -2   0   3   7 127]

Symmetric Dequantized (Float)
[-0.47244096 -0.18897638  0.          0.28346455  0.6614173  12.        ]

Asymmetric Quantized (INT8)
[-128 -122 -118 -112 -104  127]

Asymmetric Dequantized (Float)
[-0.49019608 -0.19607843  0.          0.29411766  0.6862745  12.009804  ]

Comparison
----------------------------------------------------------------------------
Symmetric Scale      : 0.09448818862438202
Symmetric Zero Point : 0
Symmetric MAE        : 0.01561680156737566
Symmetric MSE        : 0.00044051100849173963
Symmetric Max Error  : 0.038582682609558105
Symmetric Sat(min)   : 0
Symmetric Sat(max)   : 1
Symmetric Sat(total) : 1
----------------------------------------------------------------------------
Asymmetric Scale      : 0.04901960864663124
Asymmetric Zero Point : -118
Asymmetric MAE        : 0.007189512252807617
Asymmetric MSE        : 7.176663348218426e-05
Asym

In [27]:
without_outlier = np.array(
    [-0.5, -0.2, 0.0, 0.3, 0.7],
    dtype=np.float32
)

In [31]:
compare_quantization(
    "Without Outlier",
    without_outlier
)

Without Outlier :

Original Tensor
[-0.5 -0.2  0.   0.3  0.7]

Symmetric Quantized (INT8)
[-91 -36   0  54 127]

Symmetric Dequantized (Float)
[-0.5015748 -0.1984252  0.         0.2976378  0.7      ]

Asymmetric Quantized (INT8)
[-128  -64  -22   42  127]

Asymmetric Dequantized (Float)
[-0.49882355 -0.19764706  0.          0.3011765   0.7011765 ]

Comparison
----------------------------------------------------------------------------
Symmetric Scale      : 0.005511811003088951
Symmetric Zero Point : 0
Symmetric MAE        : 0.0011023670667782426
Symmetric MSE        : 2.1080245460325386e-06
Symmetric Max Error  : 0.0023622214794158936
Symmetric Sat(min)   : 0
Symmetric Sat(max)   : 1
Symmetric Sat(total) : 1
----------------------------------------------------------------------------
Asymmetric Scale      : 0.004705882631242275
Asymmetric Zero Point : -22
Asymmetric MAE        : 0.0011764795053750277
Asymmetric MSE        : 1.9377357602934353e-06
Asymmetric Max Error  : 0.002352938055

| Version | Method | Scale | Zero Pt | MAE | MSE | Max Error |
|---------|--------|------:|--------:|------:|------:|------:|
| With Outlier | Symmetric | 0.094488 | 0 | 0.015617 | 0.00044051 | 0.038583 |
| With Outlier | Asymmetric | 0.049020 | -118 | 0.007190 | 0.00007177 | 0.013725 |
| Without Outlier | Symmetric | 0.005512 | 0 | 0.001102 | 0.00000211 | 0.002362 |
| Without Outlier | Asymmetric | 0.004706 | -22 | 0.001176 | 0.00000194 | 0.002353 |

## Observations

- The outlier increased the quantization scale, leading to higher reconstruction errors.
- Removing the outlier reduced the scale and significantly improved quantization accuracy.
- Symmetric quantization performed well for values centered around zero.
- Asymmetric quantization produced lower errors when the outlier was present.